### Experiment for testing better feature vector size for the classifiers

In [1]:
# imports
from dataset_utils import *
from ClassificationHeads import BaselineClassificationHead
from sklearn.preprocessing import RobustScaler
import umap
from transformers import DistilBertTokenizer, AutoModel
import numpy as np
import torch
from pipeline_utils import create_profile_vector, create_tweet_vectors, train_classifier, test_classifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

/home/max/anaconda3/envs/social-media-bot-detection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# umap sizes to test
dim_sizes = [768, 576, 384, 192, 77, 38, 16] # 100%, 75%, 50%, 25%, 10%, 5%, 2%

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# training and testing on reduced feature vector size (ie. was passed through UMAP)
def experiment_loop(train_data_loader, validation_data_loader, dim_reducer, classifier, device = 'cuda' if torch.cuda.is_available() else 'cpu'):
    # -- initial setup
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
    scaler = RobustScaler(with_centering=False)

    # -- training loop
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(train_data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, model, max_tweets= 50, batch_size = 50)
        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i}", end="")

    # reduce tweet vectors in dimensionality
    reduced_tweet_vectors = dim_reducer.fit_transform(X=tweet_vectors, y=torch.tensor(ground_truth_labels))
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    # scale profile vectors
    scaled_profile_vectors = scaler.fit_transform(profile_vectors)
    # combine feature vectors
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, scaled_profile_vectors)]

    # perform classification training
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)

    ground_truths, predictions = train_classifier(classifier, criterion, optimizer, embedding_features, ground_truth_labels)
    print("[ -- training results -- ]")
    print("accuracy: ", accuracy_score(ground_truths, predictions))
    print("recall: ", recall_score(ground_truths, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
    print(confusion_matrix(ground_truths, predictions))

    # evaluation (meaning on dev) loop
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(validation_data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, model, max_tweets= 50, batch_size = 50)
        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i}", end="")

    # reduce tweet vectors in dimensionality
    reduced_tweet_vectors = dim_reducer.transform(X=tweet_vectors)
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    # scale profile vectors
    scaled_profile_vectors = scaler.transform(profile_vectors)

    # combine feature vectors
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, scaled_profile_vectors)]

    # perform classification testing
    ground_truths, predictions = test_classifier(classifier, embedding_features, ground_truth_labels)
    print("[ -- validation results -- ]")
    print("accuracy: ", accuracy_score(ground_truths, predictions))
    print("recall: ", recall_score(ground_truths, predictions, average="macro"))
    print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
    print(confusion_matrix(ground_truths, predictions))

In [3]:
# Dataset: Cresci 17

# datasets
dataset_mode = "train"
user_dataset = Cresci17(Cresci17SetTypes.GENUINE_USER, dataset_mode, root="./datasets", custom_label=0)
fake_dataset = Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, dataset_mode, root="./datasets", custom_label=1)
social1_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, dataset_mode, root="./datasets", custom_label=2)
social2_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, dataset_mode, root="./datasets", custom_label=3)
social3_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, dataset_mode, root="./datasets", custom_label=4)
traditional1_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, dataset_mode, root="./datasets", custom_label=5)
traditional2_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, dataset_mode, root="./datasets", custom_label=6)
traditional3_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, dataset_mode, root="./datasets", custom_label=7)
traditional4_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, dataset_mode, root="./datasets", custom_label=8)
combo_train_dataset = InterleavedIterableDataset([user_dataset, fake_dataset, social1_dataset, social2_dataset, social3_dataset,
        traditional1_dataset, traditional2_dataset, traditional3_dataset, traditional4_dataset], "Random")

dataset_mode = "dev"
user_dataset = Cresci17(Cresci17SetTypes.GENUINE_USER, dataset_mode, root="./datasets", custom_label=0)
fake_dataset = Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, dataset_mode, root="./datasets", custom_label=1)
social1_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, dataset_mode, root="./datasets", custom_label=2)
social2_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, dataset_mode, root="./datasets", custom_label=3)
social3_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, dataset_mode, root="./datasets", custom_label=4)
traditional1_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, dataset_mode, root="./datasets", custom_label=5)
traditional2_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, dataset_mode, root="./datasets", custom_label=6)
traditional3_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, dataset_mode, root="./datasets", custom_label=7)
traditional4_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, dataset_mode, root="./datasets", custom_label=8)
combo_validation_dataset = InterleavedIterableDataset([user_dataset, fake_dataset, social1_dataset, social2_dataset, social3_dataset,
        traditional1_dataset, traditional2_dataset, traditional3_dataset, traditional4_dataset], "Random")

for dim_size in dim_sizes:
    dim_reducer = umap.UMAP(n_components=dim_size, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
    classifier = BaselineClassificationHead(dim_size+7,9).to(device)
    print("[] --- experiment results for dim: " + str(dim_size) + " --- []")
    experiment_loop(combo_train_dataset, combo_validation_dataset, dim_reducer, classifier, device)

[] --- experiment results for dim: 768 --- []


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3017.33it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 8.8760
Epoch [2/15] - Loss: 1.5108
Epoch [3/15] - Loss: 1.2207
Epoch [4/15] - Loss: 1.3418
Epoch [5/15] - Loss: 1.5345
Epoch [6/15] - Loss: 1.3672
Epoch [7/15] - Loss: 1.6769
Epoch [8/15] - Loss: 0.9482
Epoch [9/15] - Loss: 1.1382
Epoch [10/15] - Loss: 1.2481
Epoch [11/15] - Loss: 1.7548
Epoch [12/15] - Loss: 1.2499
Epoch [13/15] - Loss: 1.2772
Epoch [14/15] - Loss: 1.4811
Epoch [15/15] - Loss: 1.2994
[ -- training results -- ]
accuracy:  0.8581567274519468
recall:  0.6754105395823243
f1 score:  0.6753430877509072
[[2214   61    1   11   12   23    5  130  106]
 [  71 2104    3   22   11   73    1   12   51]
 [   3    3   55    6    0    4    0    0    2]
 [  11   31    5 2689    4   18    0    4    2]
 [  15   10    2    2  299   20    0    3    5]
 [  19   80    3   21   23  615    0    4   18]
 [   8    3    0    1    0    0    0    6    6]
 [ 127   17    0    5    0    5    5   83   85]
 [  89   43    5    8    0   25   10   80  647]]
Iteration:

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9689.75it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 7.7426
Epoch [2/15] - Loss: 0.9745
Epoch [3/15] - Loss: 0.8172
Epoch [4/15] - Loss: 1.2257
Epoch [5/15] - Loss: 0.9345
Epoch [6/15] - Loss: 1.0198
Epoch [7/15] - Loss: 0.8882
Epoch [8/15] - Loss: 0.9854
Epoch [9/15] - Loss: 0.9450
Epoch [10/15] - Loss: 0.9862
Epoch [11/15] - Loss: 1.1440
Epoch [12/15] - Loss: 1.2065
Epoch [13/15] - Loss: 1.0067
Epoch [14/15] - Loss: 1.0965
Epoch [15/15] - Loss: 0.8247
[ -- training results -- ]
accuracy:  0.8669295219319862
recall:  0.6809547988079057
f1 score:  0.683721853668352
[[2263   43    4   15   24   20    2  115   77]
 [  48 2134    6   29   10   66    1    8   46]
 [   5    5   47    4    0    7    0    0    5]
 [  14   41    2 2674    2   25    1    1    4]
 [  18   12    1    1  291   25    0    2    6]
 [  20   83    3   24   20  614    1    2   16]
 [   3    0    0    0    1    0    2   11    7]
 [ 113    9    2    6    5    0   10   98   84]
 [  78   30    2   12    6   17    8   82  672]]
Iteration: 

...

Iteration: 10144Epoch [1/15] - Loss: 0.8866
Epoch [2/15] - Loss: 0.4022
Epoch [3/15] - Loss: 0.3643
Epoch [4/15] - Loss: 0.3622
Epoch [5/15] - Loss: 0.3499
Epoch [6/15] - Loss: 0.3512
Epoch [7/15] - Loss: 0.3627
Epoch [8/15] - Loss: 0.4015
Epoch [9/15] - Loss: 0.3804
Epoch [10/15] - Loss: 0.3470
Epoch [11/15] - Loss: 0.3695
Epoch [12/15] - Loss: 0.3361
Epoch [13/15] - Loss: 0.3895
Epoch [14/15] - Loss: 0.3474
Epoch [15/15] - Loss: 0.3778
[ -- training results -- ]
accuracy:  0.8875308033514047
recall:  0.7165007150455428
f1 score:  0.7219741843430769
[[2331   28    2    8    8   28    3   81   74]
 [  32 2132    5   50    3   77    0   10   39]
 [   1    7   54    1    1    5    0    1    3]
 [   8   42    3 2684    6   15    0    4    2]
 [  14   10    5    3  298   17    0    3    6]
 [  14   88    3   31   12  630    0    0    5]
 [   3    0    0    1    0    0    2    8   10]
 [ 115    9    1    2    2    3    2  119   74]
 [  38   34    0    1    6    6    4   64  754]]
Iteration:

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10139.25it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 0.6699
Epoch [2/15] - Loss: 0.3322
Epoch [3/15] - Loss: 0.2861
Epoch [4/15] - Loss: 0.2833
Epoch [5/15] - Loss: 0.2750
Epoch [6/15] - Loss: 0.2977
Epoch [7/15] - Loss: 0.2837
Epoch [8/15] - Loss: 0.2899
Epoch [9/15] - Loss: 0.2881
Epoch [10/15] - Loss: 0.2779
Epoch [11/15] - Loss: 0.2821
Epoch [12/15] - Loss: 0.2765
Epoch [13/15] - Loss: 0.2964
Epoch [14/15] - Loss: 0.2960
Epoch [15/15] - Loss: 0.2891
[ -- training results -- ]
accuracy:  0.9120749137506161
recall:  0.7392773943809157
f1 score:  0.7460208565304959
[[2342   32    3   11   10   11    4   76   74]
 [  36 2208    7   13    0   36    0    9   39]
 [   3    4   55    3    0    3    1    0    4]
 [  11   14    0 2730    3    6    0    0    0]
 [  10    2    0    6  326   10    0    1    1]
 [   9   86    5    7    6  664    0    0    6]
 [   5    0    0    0    0    0    1   12    6]
 [ 119   10    0    0    0    2    3  118   75]
 [  32   21    0    1    1    8    1   34  809]]
Iteration:

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9646.74it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 0.7225
Epoch [2/15] - Loss: 0.3369
Epoch [3/15] - Loss: 0.3132
Epoch [4/15] - Loss: 0.2983
Epoch [5/15] - Loss: 0.2910
Epoch [6/15] - Loss: 0.2847
Epoch [7/15] - Loss: 0.2861
Epoch [8/15] - Loss: 0.2761
Epoch [9/15] - Loss: 0.2795
Epoch [10/15] - Loss: 0.2840
Epoch [11/15] - Loss: 0.2891
Epoch [12/15] - Loss: 0.2918
Epoch [13/15] - Loss: 0.2769
Epoch [14/15] - Loss: 0.2871
Epoch [15/15] - Loss: 0.2861
[ -- training results -- ]
accuracy:  0.9105963528831937
recall:  0.744193151206903
f1 score:  0.7479776110665362
[[2363   30    2    9    6   18    1   72   62]
 [  34 2196    6   23    3   36    0   11   39]
 [   1    1   60    2    1    7    0    1    0]
 [  13   27    0 2714    5    3    0    0    2]
 [  10    3    1    3  333    5    0    1    0]
 [  15  100    2   14    3  641    0    0    8]
 [   0    1    0    0    0    0    0   20    3]
 [ 101    8    0    3    0    4    0  131   80]
 [  19   28    0    9    1   17    1   32  800]]
Iteration: 

In [4]:
# Dataset: Cresci 18:
train_dataset = Cresci18("train", root="./datasets", use_only_labelled=True, label_mapping=[1,0,"unlabelled"])
validation_dataset = Cresci18("dev", root="./datasets", use_only_labelled=True, label_mapping=[1,0,"unlabelled"])

for dim_size in dim_sizes:
    dim_reducer = umap.UMAP(n_components=dim_size, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
    classifier = BaselineClassificationHead(dim_size+7,9).to(device)
    print("[] --- experiment results for dim: " + str(dim_size) + " --- []")
    experiment_loop(combo_train_dataset, combo_validation_dataset, dim_reducer, classifier, device)

[] --- experiment results for dim: 768 --- []


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8758.20it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 9.6550
Epoch [2/15] - Loss: 0.9472
Epoch [3/15] - Loss: 1.3532
Epoch [4/15] - Loss: 0.9230
Epoch [5/15] - Loss: 1.1338
Epoch [6/15] - Loss: 1.1582
Epoch [7/15] - Loss: 1.8455
Epoch [8/15] - Loss: 1.2721
Epoch [9/15] - Loss: 1.1475
Epoch [10/15] - Loss: 2.3418
Epoch [11/15] - Loss: 1.5081
Epoch [12/15] - Loss: 1.4251
Epoch [13/15] - Loss: 1.2345
Epoch [14/15] - Loss: 1.4184
Epoch [15/15] - Loss: 1.3771
[ -- training results -- ]
accuracy:  0.8417939871858058
recall:  0.6704219068570849
f1 score:  0.6698660259431076
[[2198   60    5   17   15   35    4  132   97]
 [  53 2059    5   45   24   90    0   17   55]
 [   1    5   59    2    0    4    0    1    1]
 [  10   47    1 2659    1   42    0    2    2]
 [  31    5    2    5  272   23    1    6   11]
 [  32  119    2   27   25  561    0    3   14]
 [   8    1    0    0    0    0    2    4    9]
 [ 126   17    1    3    4    5    7   82   82]
 [ 101   36    0    7   10   15    9   81  648]]
Iteration:

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9247.93it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 6.1470
Epoch [2/15] - Loss: 0.8998
Epoch [3/15] - Loss: 1.3332
Epoch [4/15] - Loss: 0.7673
Epoch [5/15] - Loss: 0.8089
Epoch [6/15] - Loss: 0.9368
Epoch [7/15] - Loss: 1.0351
Epoch [8/15] - Loss: 0.8114
Epoch [9/15] - Loss: 0.9697
Epoch [10/15] - Loss: 1.0815
Epoch [11/15] - Loss: 0.9847
Epoch [12/15] - Loss: 1.0066
Epoch [13/15] - Loss: 1.0204
Epoch [14/15] - Loss: 1.6051
Epoch [15/15] - Loss: 1.1772
[ -- training results -- ]
accuracy:  0.866436668309512
recall:  0.6669693751002426
f1 score:  0.668444545151245
[[2268   46    4    8   17   22    5  103   90]
 [  52 2123    9   19   13   61    4   15   52]
 [   3    6   48    2    0    9    1    1    3]
 [   9   21    1 2712    3   17    0    1    0]
 [  18    7    1    1  277   42    0    5    5]
 [  24   87    6   18   36  593    0    2   17]
 [   4    2    0    0    0    1    0    8    9]
 [ 118   25    0    3    0    2    3   98   78]
 [  69   39    3    5    0   35    6   79  671]]
Iteration: 1

...

Iteration: 10144Epoch [1/15] - Loss: 0.7663
Epoch [2/15] - Loss: 0.3566
Epoch [3/15] - Loss: 0.3273
Epoch [4/15] - Loss: 0.3257
Epoch [5/15] - Loss: 0.3092
Epoch [6/15] - Loss: 0.3164
Epoch [7/15] - Loss: 0.3150
Epoch [8/15] - Loss: 0.2993
Epoch [9/15] - Loss: 0.3147
Epoch [10/15] - Loss: 0.3294
Epoch [11/15] - Loss: 0.3044
Epoch [12/15] - Loss: 0.3518
Epoch [13/15] - Loss: 0.3247
Epoch [14/15] - Loss: 0.3374
Epoch [15/15] - Loss: 0.3367
[ -- training results -- ]
accuracy:  0.902513553474618
recall:  0.7375386702841954
f1 score:  0.7410964261071855
[[2333   35    3    7    9   11    4   93   68]
 [  36 2181    0   28    6   42    1    5   49]
 [   3    1   61    0    6    2    0    0    0]
 [  14   33    1 2689    1   26    0    0    0]
 [  11    1    3    4  316   19    0    1    1]
 [  10   41    1   45   14  668    0    0    4]
 [   8    1    0    0    0    0    0    9    6]
 [ 105   21    0    2    0    0    5  127   67]
 [  42   33    0    2    1    4    2   42  781]]
Iteration: 

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9118.45it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 0.7876
Epoch [2/15] - Loss: 0.3704
Epoch [3/15] - Loss: 0.3098
Epoch [4/15] - Loss: 0.2951
Epoch [5/15] - Loss: 0.3096
Epoch [6/15] - Loss: 0.3086
Epoch [7/15] - Loss: 0.2837
Epoch [8/15] - Loss: 0.3005
Epoch [9/15] - Loss: 0.2881
Epoch [10/15] - Loss: 0.3188
Epoch [11/15] - Loss: 0.3021
Epoch [12/15] - Loss: 0.3133
Epoch [13/15] - Loss: 0.3051
Epoch [14/15] - Loss: 0.2941
Epoch [15/15] - Loss: 0.2905
[ -- training results -- ]
accuracy:  0.9078363725973386
recall:  0.7390904391594157
f1 score:  0.7459206261382219
[[2359   27    3    6    4   21    1   73   69]
 [  31 2181    5   41    2   39    0    4   45]
 [   2    3   59    3    2    4    0    0    0]
 [   7   39    1 2693    1   23    0    0    0]
 [   8    1    0    9  326   12    0    0    0]
 [   8   76    0   34    7  655    0    0    3]
 [   4    1    0    1    0    0    0   11    7]
 [ 112    5    0    5    0    2    5  120   78]
 [  32   19    0   10    1    4    1   23  817]]
Iteration:

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8860.14it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144Epoch [1/15] - Loss: 0.6014
Epoch [2/15] - Loss: 0.3259
Epoch [3/15] - Loss: 0.2953
Epoch [4/15] - Loss: 0.2911
Epoch [5/15] - Loss: 0.2964
Epoch [6/15] - Loss: 0.2890
Epoch [7/15] - Loss: 0.2947
Epoch [8/15] - Loss: 0.2881
Epoch [9/15] - Loss: 0.2957
Epoch [10/15] - Loss: 0.2927
Epoch [11/15] - Loss: 0.2813
Epoch [12/15] - Loss: 0.2893
Epoch [13/15] - Loss: 0.2856
Epoch [14/15] - Loss: 0.2865
Epoch [15/15] - Loss: 0.2771
[ -- training results -- ]
accuracy:  0.9117792015771315
recall:  0.7532394514141132
f1 score:  0.7555931968698131
[[2398   18    3    9    6   20    0   52   57]
 [  29 2151    1   41    5   75    1    7   38]
 [   1    3   63    1    0    5    0    0    0]
 [   8   41    1 2698    1   10    0    3    2]
 [   9    7    0    1  332    5    0    0    2]
 [  12   74    4   38    2  647    0    0    6]
 [   8    0    0    1    0    0    0   10    5]
 [ 103   14    1    3    0    1    2  138   65]
 [  15   14    1    3    0   11    0   40  823]]
Iteration:

In [ ]:
# Dataset: Caverlee 11:
train_dataset = Cresci18("train", root="./datasets", use_only_labelled=True, label_mapping=[1,0,"unlabelled"])
validation_dataset = Cresci18("dev", root="./datasets", use_only_labelled=True, label_mapping=[1,0,"unlabelled"])

for dim_size in dim_sizes:
    dim_reducer = umap.UMAP(n_components=dim_size, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
    classifier = BaselineClassificationHead(dim_size+7,9).to(device)
    print("[] --- experiment results for dim: " + str(dim_size) + " --- []")
    experiment_loop(combo_train_dataset, combo_validation_dataset, dim_reducer, classifier, device)

In [ ]:
# Dataset: Twibot20
train_dataset = Twibot20("train", root="./datasets", label_mapping=[1,0])
validation_dataset = Twibot20("dev", root="./datasets", label_mapping=[1,0])

for dim_size in dim_sizes:
    dim_reducer = umap.UMAP(n_components=dim_size, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
    classifier = BaselineClassificationHead(dim_size+7,9).to(device)
    print("[] --- experiment results for dim: " + str(dim_size) + " --- []")
    experiment_loop(combo_train_dataset, combo_validation_dataset, dim_reducer, classifier, device)

In [ ]:
# Dataset: Twibot22
train_dataset = Twibot22("train", root="./datasets", label_mapping=[1,0])
validation_dataset = Twibot22("dev", root="./datasets", label_mapping=[1,0])

for dim_size in dim_sizes:
    dim_reducer = umap.UMAP(n_components=dim_size, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
    classifier = BaselineClassificationHead(dim_size+7,9).to(device)
    print("[] --- experiment results for dim: " + str(dim_size) + " --- []")
    experiment_loop(combo_train_dataset, combo_validation_dataset, dim_reducer, classifier, device)